In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

result_path = Path("../results/onnx_cpu_fp32.jsonl")

# Load JSONL and expand nested machine_metadata
records = pd.read_json(result_path, lines=True)
if "machine_metadata" in records.columns:
    meta_df = pd.json_normalize(records["machine_metadata"])
    meta_df.columns = [f"machine_meta_{c}" for c in meta_df.columns]
    records = pd.concat([records, meta_df], axis=1)
    records = records.drop(columns=["machine_metadata"])

display(records.head())


In [ ]:
# Machine metadata summary
machine_meta_cols = [
    "machine_meta_host_hostname",
    "machine_meta_cpu_model_name",
    "machine_meta_cpu_vendor_id",
    "machine_meta_cpu_architecture",
    "machine_meta_cpu_physical_cores",
    "machine_meta_cpu_logical_cores",
    "machine_meta_cpu_max_mhz",
    "machine_meta_cpu_current_mhz",
    "machine_meta_cpu_features_avx",
    "machine_meta_cpu_features_avx2",
    "machine_meta_cpu_features_avx512",
    "machine_meta_cpu_features_avx512_vnni",
    "machine_meta_cpu_features_amx_bf16",
    "machine_meta_cpu_features_amx_int8",
    "machine_meta_memory_total_bytes",
    "machine_meta_container_is_container",
    "machine_meta_kubernetes_is_kubernetes",
    "machine_meta_gpu_available",
    "machine_meta_gpu_count",
]

available_meta_cols = [c for c in machine_meta_cols if c in records.columns]
display(records[available_meta_cols].drop_duplicates())


In [ ]:
cols = [
    "dataset",
    "batch_size",
    "max_length",
    "avg_tokens_per_item",
    "tokenize_tokens_per_sec",
    "embedding_tokens_per_sec",
    "end_to_end_tokens_per_sec",
    "tokenize_items_per_sec",
    "embedding_items_per_sec",
    "end_to_end_items_per_sec",
    "tokenize_latency_ms_p95",
    "embedding_latency_ms_p95",
    "end_to_end_latency_ms_p95",
]

display(records[cols].sort_values(["dataset", "max_length", "batch_size"]))


In [ ]:
plot_df = records.copy()
plot_df["run"] = (
    plot_df["dataset"].astype(str)
    + ", bs=" + plot_df["batch_size"].astype(str)
    + ", len=" + plot_df["max_length"].astype(str)
)

metrics = [
    "tokenize_tokens_per_sec",
    "embedding_tokens_per_sec",
    "end_to_end_tokens_per_sec",
]

ax = plot_df.plot.bar(
    x="run",
    y=metrics,
    figsize=(16, 6),
)

ax.set_title("BGE-M3 ONNX CPU FP32: Tokenization vs Embedding Throughput")
ax.set_xlabel("Benchmark run")
ax.set_ylabel("tokens/sec")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
latency_metrics = [
    "tokenize_latency_ms_p95",
    "embedding_latency_ms_p95",
    "end_to_end_latency_ms_p95",
]

ax = plot_df.plot.bar(
    x="run",
    y=latency_metrics,
    figsize=(16, 6),
)

ax.set_title("BGE-M3 ONNX CPU FP32: p95 Latency Breakdown")
ax.set_xlabel("Benchmark run")
ax.set_ylabel("p95 latency ms")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
records["tokenization_overhead_ratio"] = (
    records["tokenize_latency_ms_p95"]
) / records["end_to_end_latency_ms_p95"]

plot_df = records.copy()
plot_df["run"] = (
    plot_df["dataset"].astype(str)
    + ", bs=" + plot_df["batch_size"].astype(str)
    + ", len=" + plot_df["max_length"].astype(str)
)

ax = plot_df.plot.bar(
    x="run",
    y="tokenization_overhead_ratio",
    figsize=(16, 5),
    legend=False,
)

ax.set_title("BGE-M3 ONNX CPU FP32: Tokenization Overhead Ratio")
ax.set_xlabel("Benchmark run")
ax.set_ylabel("ratio")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
group_cols = [
    "machine_meta_cpu_model_name",
    "machine_meta_cpu_physical_cores",
    "machine_meta_cpu_logical_cores",
    "machine_meta_cpu_features_avx2",
    "machine_meta_cpu_features_avx512",
    "dataset",
    "batch_size",
    "max_length",
]

available_group_cols = [c for c in group_cols if c in records.columns]

summary = (
    records.groupby(available_group_cols, as_index=False, dropna=False)
      .agg(
          tokenize_tokens_per_sec=("tokenize_tokens_per_sec", "median"),
          embedding_tokens_per_sec=("embedding_tokens_per_sec", "median"),
          end_to_end_tokens_per_sec=("end_to_end_tokens_per_sec", "median"),
          end_to_end_latency_ms_p95=("end_to_end_latency_ms_p95", "median"),
      )
)

display(summary)


In [ ]:
for dataset in sorted(summary["dataset"].unique()):
    subset = summary[summary["dataset"] == dataset]

    pivot = subset.pivot_table(
        index="batch_size",
        columns="max_length",
        values="embedding_tokens_per_sec",
    )

    ax = pivot.plot(figsize=(10, 5), marker="o")
    ax.set_title(f"BGE-M3 ONNX CPU FP32: Embedding tokens/sec - dataset={dataset}")
    ax.set_xlabel("batch_size")
    ax.set_ylabel("embedding_tokens_per_sec")
    plt.tight_layout()
    plt.show()
